<a href="https://colab.research.google.com/github/miraynurasma/LivArea-Tubitak-2209/blob/main/Sehir_Yasanabilirlik_Analizi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# Dosyayı hem noktalı virgül ayracıyla hem de doğru karakter setiyle okuyoruz
dosya_yolu = 'veri.csv'
df = pd.read_csv(dosya_yolu, sep=';', encoding='latin-1')

# Sütun isimlerindeki o tuhaf karakterleri temizleyelim
df.columns = df.columns.str.replace('Ä±', 'i').str.replace('ÄŸ', 'g').str.replace('Ä', 'A')

# Şehir isimlerini düzeltelim (AdÄ±yaman -> Adıyaman gibi)
# Not: Bu aşamada sütun isminin tam halini görmemiz lazım.
# Önce yukarıdaki satırı çalıştırıp tabloyu görelim.

df.head()

In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# 1. 'veri.csv' yerine halihazırda var olan 'analiz_edildi.csv' dosyasını okuyoruz
try:
    df = pd.read_csv('/content/analiz_edildi.csv')
    print("✅ Mevcut dosya başarıyla okundu.")
except:
    print("⚠️ Dosya bulunamadı! Lütfen orijinal verini sol tarafa yükle.")

# 2. YAPAY ZEKA PUANLAMASI (Garantiye alalım)
features = ["ortalama_kira_2024", "issizlik_oranı_2024", "hava_kalitesi_pm25_2024", "nufus_yogunlugu"]
scaler = MinMaxScaler()
df[features] = df[features].fillna(df[features].mean()) # Boş veri varsa doldurur
scaled = scaler.fit_transform(df[features])

# Yaşam Skoru Hesaplama
df["yasam_skoru"] = ((1-scaled[:,0])*0.3 + (1-scaled[:,1])*0.3 + (1-scaled[:,2])*0.2 + (1-scaled[:,3])*0.2) * 100

# 3. Dosyayı tekrar kaydediyoruz
df.to_csv('analiz_edildi.csv', index=False)
print("✅ Puanlar hesaplandı ve dosya güncellendi!")

In [ ]:
df.info()

In [ ]:
# 1. Nüfus Yoğunluğu
df['nufus_yogunlugu'] = df['nufus_2024'] / df['yuzolcumu_km2']

# 2. Araç Yoğunluğu
df['arac_yogunlugu'] = df['arac_sayısı_2024'] / df['yuzolcumu_km2']

# 3. Kira Artış Oranı
df['kira_artis_orani'] = (df['ortalama_kira_2024'] - df['ortalama_kira_2023']) / df['ortalama_kira_2023']

# 4. Kişi Başına Düşen Araç Sayısı
df['kisi_basi_arac'] = df['arac_sayısı_2024'] / df['nufus_2024']

# Tabloyu kontrol et
df[['il_adı', 'nufus_yogunlugu', 'kira_artis_orani', 'arac_yogunlugu']].head()

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# 1. Analize dahil etmek istediğimiz sayısal sütunları seçiyoruz
features = ['nufus_yogunlugu', 'kira_artis_orani', 'arac_yogunlugu',
            'issizlik_oranı_2024', 'hava_kalitesi_pm25_2024']

x = df[features]

# 2. Verileri ölçeklendiriyoruz (Standartlaştırma)
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)

# 3. Modeli kuralım (Şehirleri 4 farklı gruba ayır diyoruz)
kmeans = KMeans(n_clusters=4, random_state=42)
df['il_grubu'] = kmeans.fit_predict(x_scaled)

# Hangi il hangi gruba düşmüş görelim
df[['il_adı', 'il_grubu']].head(10)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Grupları görselleştirelim (Kira Artış Oranı ve Nüfus Yoğunluğu bazında)
plt.figure(figsize=(12, 8))
sns.scatterplot(data=df, x='nufus_yogunlugu', y='kira_artis_orani',
                hue='il_grubu', palette='viridis', s=100)

plt.title('Şehirlerin Gruplara Göre Dağılımı (Kira vs Nüfus Yoğunluğu)')
plt.xlabel('Nüfus Yoğunluğu')
plt.ylabel('Kira Artış Oranı')
plt.legend(title='İl Grubu')
plt.grid(True)
plt.show()

In [ ]:
# Analiz edilmiş veriyi yeni bir CSV olarak kaydedelim
df.to_csv('analiz_edildi.csv', index=False)

# Dosyayı bilgisayarına indirmek için (Colab hilesi)
from google.colab import files
files.download('analiz_edildi.csv')

In [ ]:
!pip install streamlit plotly pyngrok

In [ ]:
# 1. Önce şifreni (Endpoint IP) buraya yazdırıyoruz
print("Şifreniz (Endpoint IP):")
!curl ipv4.icanhazip.com

# 2. Şimdi uygulamayı ve linki (URL) çalıştırıyoruz
!streamlit run app.py & npx localtunnel --port 8501

In [ ]:
# 1. Her şeyi sıfırlayalım
!pkill streamlit

# 2. IP Adresini yine alalım (Şifren olacak)
print("🔑 Endpoint IP Adresin:")
!curl ipv4.icanhazip.com

# 3. Streamlit'i en güvenli ayarlarla başlatalım
!streamlit run app.py --server.port 8501 --server.enableCORS=false --server.enableXsrfProtection=false & npx localtunnel --port 8501

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd

# 1. EN HIZLI VE HATASIZ SAYFA AYARI
st.set_page_config(page_title="Şehir Analizi", layout="centered")

# 2. VERİYİ YÜKLE
df = pd.read_csv('/content/analiz_edildi.csv')

# 3. BAŞLIK
st.title("🏙️ Şehir Yaşanabilirlik Analizi")
st.info("Yapay Zeka Destekli Şehir Değerlendirme Sistemi")

# 4. ŞEHİR SEÇİMİ (Bu kısım asla hata vermez)
secilen_il = st.selectbox("Analiz etmek istediğiniz şehri seçin:", df["il_adı"].sort_values())
veri = df[df["il_adı"] == secilen_il].iloc[0]

# 5. SKOR VE ANALİZ SONUCU
st.divider()
st.subheader(f"📊 {secilen_il} İli Analiz Sonucu")

# Büyük Skor Gösterimi
skor = veri['yasam_skoru']
st.metric(label="Yapay Zeka Yaşam Skoru", value=f"{skor:.1f} / 100")

# Akıllı Değerlendirme Kartı
if skor > 65:
    st.success("🟢 BU ŞEHİR: Yüksek yaşam kalitesi ve dengeli imkanlar sunuyor.")
elif skor > 45:
    st.warning("🟡 BU ŞEHİR: Gelişme potansiyeli yüksek, orta seviye yaşam şartları.")
else:
    st.error("🔴 BU ŞEHİR: Yaşam maliyeti veya çevre koşulları açısından iyileştirme gerektiriyor.")

# 6. VERİ TABLOSU (Harita yerine bunu koyuyoruz, hata vermesi imkansız!)
st.write("---")
st.subheader("📋 Detaylı Veriler")
st.dataframe(df[df["il_adı"] == secilen_il][["ortalama_kira_2024", "issizlik_oranı_2024", "hava_kalitesi_pm25_2024"]])

st.caption("Miray tarafından TÜBİTAK projesi kapsamında geliştirilmiştir.")

In [ ]:
import pandas as pd
df = pd.read_csv('analiz_edildi.csv')
print(df.head()) # İlk 5 satırı görmeni sağlar

In [ ]:
import pandas as pd

# 1. Dosyayı oku
df = pd.read_csv('analiz_edildi.csv')

# 2. Min-Max Normalizasyonu (0-100 arası)
# Kural: Kira ve Hava Kirliliği 'Negatif' kriterdir (Düşük olması iyidir)
def normalize_negatif(seri):
    return 100 * (1 - (seri - seri.min()) / (seri.max() - seri.min()))

# Sütunları skorlayalım
df['kira_skor'] = normalize_negatif(df['ortalama_kira_2024'])
df['hava_skor'] = normalize_negatif(df['hava_kalitesi_pm25_2024'])
df['issizlik_skor'] = normalize_negatif(df['issizlik_oranı_2022'])

# 3. Bütünleşik Yaşam Skoru (Ağırlıklı Ortalama)
# Şimdilik eşit dağıtalım, istersen kirayı %40 yapabiliriz!
df['skor'] = (df['kira_skor'] + df['hava_skor'] + df['issizlik_skor']) / 3

# 4. Kaydedelim (Harita bu dosyayı okuyacak)
df.to_csv('harita_verisi.csv', index=False)
print("✅ Dün geceki skorlar başarıyla hesaplandı ve 'skor' sütunu eklendi!")

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import folium
from streamlit_folium import st_folium
import json

st.set_page_config(layout="wide", page_title="LivArea Analitik Portal")

# Veriyi oku
try:
    df = pd.read_csv('harita_verisi_final.csv')
except:
    df = pd.read_csv('harita_verisi.csv')

st.sidebar.title("🏙️ LivArea")
st.sidebar.subheader("TÜBİTAK 2209-A")
yil = st.sidebar.selectbox("Yıl Seçiniz", [2022, 2023, 2024])

st.title(f"Kentsel Yaşam Kalitesi Analitik Portalı ({yil})")

# HARİTA AYARLARI
# Harita dosyasını yerel olarak çekmeye zorluyoruz
geojson_url = "https://raw.githubusercontent.com/cihadturhan/tr-geojson/master/tr-cities.json"

m = folium.Map(location=[39.0, 35.0], zoom_start=6, tiles="CartoDB positron")

folium.Choropleth(
    geo_data=geojson_url,
    name="choropleth",
    data=df,
    columns=["il_adı", "skor"],
    key_on="feature.properties.name",
    fill_color="YlGn",
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Yaşanabilirlik Puanı"
).add_to(m)

# Haritayı ekrana bas (Bu sefer daha güvenli bir yöntem)
st_folium(m, width=900, height=500)

# Alt kısım
secilen_il = st.selectbox("İncelemek İçin İl Seçiniz", df['il_adı'].unique())
il_veri = df[df['il_adı'] == secilen_il].iloc[0]
st.info(f"**{secilen_il}** Puanı: {il_veri['skor']:.2f}")

In [ ]:
!npm install -g localtunnel

In [ ]:
!streamlit run app.py & npx localtunnel --port 8501


In [ ]:
!pip install streamlit

In [ ]:
import pandas as pd

df = pd.read_csv('harita_verisi.csv')

# İsimlerdeki boşlukları temizleyelim ve ilk harfleri büyütelim
df['il_adı'] = df['il_adı'].str.strip().str.title()

# Harita dosyasındaki yaygın isim farklarını düzeltelim
duzeltmeler = {
    "Afyon": "Afyonkarahisar",
    "İçel": "Mersin"
}
df['il_adı'] = df['il_adı'].replace(duzeltmeler)

# Dosyayı tekrar kaydedelim
df.to_csv('harita_verisi_final.csv', index=False)
print("✅ İl isimleri haritaya uygun hale getirildi!")

In [ ]:
import pandas as pd

# Veriyi oku
df = pd.read_csv('harita_verisi.csv')

# İsimleri temizle: Boşlukları at, hepsini Baş Harfi Büyük yap
df['il_adı'] = df['il_adı'].str.strip().str.title()

# Harita dosyasındaki (GeoJSON) özel yazımlara göre eşitleme
duzeltmeler = {
    "Afyon": "Afyonkarahisar",
    "İçel": "Mersin",
    "K.Maras": "Kahramanmaraş",
    "G.Antep": "Gaziantep"
}
df['il_adı'] = df['il_adı'].replace(duzeltmeler)

# Yeni dosyayı kaydet
df.to_csv('harita_verisi_final.csv', index=False)
print("✅ 81 ilin ismi harita için mühürlendi!")

In [ ]:
!curl ipv4.icanhazip.com

In [ ]:
!curl ipv4.icanhazip.com

In [ ]:
!pip install streamlit-folium


In [ ]:
!curl ipv4.icanhazip.com

In [ ]:
!wget https://raw.githubusercontent.com/cihadturhan/tr-geojson/master/tr-cities.json -O turkiye.json